# Week 4 — Customer Acquisition Cost (CAC)

Neither entity discloses a clean, auditable CAC:
- **HDFC Bank**: listed banks don't report per-customer CAC — confirmed absence, not a gap in our collection.
- **Jupiter**: only a founder-attributed anecdotal claim ($5–6/customer), not RoC-disclosed or audited.

So this notebook does **not** manufacture a precise CAC for either side. Instead it computes
a transparent, clearly-labeled **ad-spend intensity proxy** (`advertising spend / disclosed
customer base`) from officially disclosed ad-spend and customer-base figures, and separately
carries forward the anecdotal Jupiter CAC claim with its confidence flag intact.

Figures sourced from `data/raw/cac_signals.md` (Entrackr/Inc42/Storyboard18/Tracxn coverage,
accessed 2026 — see that file for full citations and caveats).

**Output:** `data/processed/cac_signals_clean.csv`

In [1]:
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent
PROCESSED = ROOT / "data" / "processed"

# Transcribed from data/raw/cac_signals.md — see that file for sources/confidence per row.
rows = [
    {"entity": "Jupiter", "metric": "Advertising & promotional expense FY23", "value": 74.5, "unit": "INR crore", "confidence": "official (RoC filing via press)"},
    {"entity": "Jupiter", "metric": "Users, ~10 months post-launch", "value": 1_000_000, "unit": "users", "confidence": "unverified/approximate"},
    {"entity": "Jupiter", "metric": "Users, Oct 2025 (conflicting figure)", "value": 300_000, "unit": "users", "confidence": "unverified/approximate"},
    {"entity": "Jupiter", "metric": "Claimed CAC (founder-attributed)", "value": 5.5, "unit": "USD/customer (midpoint of $5-6)", "confidence": "low-confidence/anecdotal"},
    {"entity": "HDFC Bank", "metric": "Advertising & publicity expenditure FY24", "value": 1259.35, "unit": "INR crore", "confidence": "official"},
    {"entity": "HDFC Bank", "metric": "Total customer base (post-merger)", "value": 120_000_000, "unit": "customers", "confidence": "official"},
    {"entity": "HDFC Bank", "metric": "Disclosed per-customer CAC", "value": None, "unit": "USD/customer", "confidence": "confirmed absence"},
]
cac_df = pd.DataFrame(rows)
cac_df

,entity,metric,value,unit,confidence
0,Jupiter,Advertising & promotional expense FY23,7.450000e+01,INR crore,official (RoC filing via press)
1,Jupiter,"Users, ~10 months post-launch",1.000000e+06,users,unverified/approximate
2,Jupiter,"Users, Oct 2025 (conflicting figure)",3.000000e+05,users,unverified/approximate
3,Jupiter,Claimed CAC (founder-attributed),5.500000e+00,USD/customer (midpoint of $5-6),low-confidence/anecdotal
4,HDFC Bank,Advertising & publicity expenditure FY24,1.259350e+03,INR crore,official
5,HDFC Bank,Total customer base (post-merger),1.200000e+08,customers,official
6,HDFC Bank,Disclosed per-customer CAC,NaN,USD/customer,confirmed absence


## Ad-spend intensity proxy

`ad spend (INR crore) x 1e7 / customer base` in INR/customer — **not a true CAC**
(the denominator is the *whole* disclosed customer base, not new customers acquired that
year), but it's the most defensible number computable from what's actually disclosed on
both sides. Jupiter's is shown for both conflicting user-count figures given the source
disagreement flagged in `cac_signals.md`.

In [2]:
def spend_intensity(ad_spend_cr, customers):
    return (ad_spend_cr * 1e7) / customers

hdfc_intensity = spend_intensity(1259.35, 120_000_000)
jupiter_intensity_high = spend_intensity(74.5, 1_000_000)
jupiter_intensity_low = spend_intensity(74.5, 300_000)

print(f"HDFC Bank: INR {hdfc_intensity:,.0f} of ad spend per disclosed customer")
print(f"Jupiter (vs. ~1M users): INR {jupiter_intensity_high:,.0f} of ad spend per disclosed user")
print(f"Jupiter (vs. ~300K users): INR {jupiter_intensity_low:,.0f} of ad spend per disclosed user")

HDFC Bank: INR 105 of ad spend per disclosed customer
Jupiter (vs. ~1M users): INR 745 of ad spend per disclosed user
Jupiter (vs. ~300K users): INR 2,483 of ad spend per disclosed user


In [3]:
proxy_rows = pd.DataFrame([
    {"entity": "HDFC Bank", "ad_spend_intensity_inr_per_customer": round(hdfc_intensity, 2), "basis": "FY24 ad spend / ~12cr total customers", "confidence": "proxy — not true CAC"},
    {"entity": "Jupiter", "ad_spend_intensity_inr_per_customer": round(jupiter_intensity_high, 2), "basis": "FY23 ad spend / ~1M users (10mo post-launch estimate)", "confidence": "proxy — not true CAC"},
    {"entity": "Jupiter", "ad_spend_intensity_inr_per_customer": round(jupiter_intensity_low, 2), "basis": "FY23 ad spend / ~300K users (Oct 2025 conflicting estimate)", "confidence": "proxy — not true CAC"},
])

cac_out = pd.concat([cac_df, proxy_rows], ignore_index=True)
cac_out.to_csv(PROCESSED / "cac_signals_clean.csv", index=False)
print("-> data/processed/cac_signals_clean.csv")
cac_out

-> data/processed/cac_signals_clean.csv


,entity,metric,value,unit,confidence,ad_spend_intensity_inr_per_customer,basis
0,Jupiter,Advertising & promotional expense FY23,7.450000e+01,INR crore,official (RoC filing via press),NaN,NaN
1,Jupiter,"Users, ~10 months post-launch",1.000000e+06,users,unverified/approximate,NaN,NaN
2,Jupiter,"Users, Oct 2025 (conflicting figure)",3.000000e+05,users,unverified/approximate,NaN,NaN
3,Jupiter,Claimed CAC (founder-attributed),5.500000e+00,USD/customer (midpoint of $5-6),low-confidence/anecdotal,NaN,NaN
4,HDFC Bank,Advertising & publicity expenditure FY24,1.259350e+03,INR crore,official,NaN,NaN
5,HDFC Bank,Total customer base (post-merger),1.200000e+08,customers,official,NaN,NaN
6,HDFC Bank,Disclosed per-customer CAC,NaN,USD/customer,confirmed absence,NaN,NaN
7,HDFC Bank,NaN,NaN,NaN,proxy — not true CAC,104.95,FY24 ad spend / ~12cr total customers
8,Jupiter,NaN,NaN,NaN,proxy — not true CAC,745.00,FY23 ad spend / ~1M users (10mo post-launch es...
9,Jupiter,NaN,NaN,NaN,proxy — not true CAC,2483.33,FY23 ad spend / ~300K users (Oct 2025 conflict...


**Composite-index note:** CAC is intentionally *excluded* from the Week 4 composite index
(see `05_composite_index.ipynb`) — HDFC has no disclosed CAC to compare against, and the two
ad-spend-intensity proxies aren't measuring the same thing (whole customer base vs. new
acquisitions), so folding them into one score would manufacture false precision.